In [ ]:
%matplotlib ipympl

from helper import *
import ipywidgets
import scipy.linalg as sci_lin

## Taylor plus LQR

We use a Taylor series expansion for the first 0.5s, and then we use an LQR to regulate for the remaining 1.5s.

In [ ]:
alpha = np.eye(3) * -1.5
A = np.array([[0, 1, 0], [0, 0, 1], [0, 0, 0]], dtype=float)
B = np.array([[0], [0], [1]], dtype=float)
Q = np.diag([1e0, 1e0, 1e0])
R = np.array([[1e-0]], dtype=float)
K, _, _ = ct.lqr(A - alpha, B, Q, R)
E = sci_lin.expm((A - B @ K) * spec.dt)

In [ ]:
@functools.partial(jax.jit, static_argnames=["n"])
def propogate_pos(x0, E, n):
    def scan_body(x0, _):
        x1 = E @ x0
        return x1, x1[0]

    _, res = jax.lax.scan(scan_body, x0, length=n)
    return res.flatten()

def diff(x):
    return jnp.diff(x) / spec.dt

def taylor_coeffs(hist):
    assert hist.shape == (4,)
    a0 = hist[3]
    a1 = diff(hist[2:])
    a2 = diff(diff(hist[1:])) / 2
    a3 = diff(diff(diff(hist))) / 6
    return a0, a1, a2, a3

def taylor_eval(hist, x):
    a0, a1, a2, a3 = taylor_coeffs(hist)
    return a0 + a1 * x + a2 * x**2 + a3 * x**3

def taylor_evalp(hist, x):
    a0, a1, a2, a3 = taylor_coeffs(hist)
    f0 = a0 + a1 * x + a2 * x**2 + a3 * x**3
    f1 = a1 + 2 * a2 * x + 3 * a3 * x**2
    f2 = 2 * a2 + 6 * a3 * x
    return jnp.array([f0, f1, f2])

def mixed_pred(hist):
    assert spec.n == 200
    assert spec.dt == 0.01
    n_taylor = 50
    t = jnp.arange(1, n_taylor + 1, dtype=float) * spec.dt
    res0 = taylor_eval(hist, t)
    x0 = taylor_evalp(hist, t[-1])
    res1 = propogate_pos(x0, E, 200 - n_taylor)
    res = jnp.concatenate([res0, res1])
    assert res.shape == (200,)
    return res

In [ ]:
hist = y_vest[:4, 0]
y = mixed_pred(hist)
fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(y)
ax.plot(y_vest[4: 4 + 200, 0])
ax.grid()

## quantitative analysis

In [ ]:
lti_int, x_data, y_data, ctrl_data = get_data(0)

In [ ]:
def pred_fun(hist, t, _):
    return mixed_pred(hist)

pred_err(pred_fun, y_data, 4)

In [ ]:
def pred_fun_check(hist, t, idx):
    x0 = x_data[idx]
    _, y = lti_int(x0=x0, u=jnp.ones_like(t) * ctrl_data[idx])
    return y

pred_err(pred_fun_check, y_data, 2)

## visualize

In [ ]:
idx = 4442
t = np.linspace(0, 2.0, num=spec.n + 1, endpoint=True)

fig, axs = plt.subplots(2, 1, figsize=(7, 8))
axs[0].plot(y_data[idx + 1: idx + spec.n + 2], label="y_data")
axs[0].plot(pred_fun(y_data[idx - 3: idx + 1], t, idx), label="pred")
axs[1].plot(y_data[idx + 1: idx + spec.n + 2], label="y_data")
axs[1].plot(pred_fun_check(y_data[idx - 1: idx + 1], t, idx), label="flat_pred")
for ax in axs:
    ax.grid()
    ax.legend()

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(7, 8))

idx = 1000
t = np.linspace(0, 2.0, num=spec.n + 1, endpoint=True)
line0, = axs[0].plot(y_data[idx + 1: idx + spec.n + 2], label="y_data")
line1, = axs[0].plot(pred_fun(y_data[idx - 3: idx + 1], t, idx), label="pred")
line2, = axs[1].plot(y_data[idx + 1: idx + spec.n + 2], label="y_data")
line3, = axs[1].plot(pred_fun_check(y_data[idx - 1: idx + 1], t, idx), label="flat_pred")
for ax in axs:
    ax.grid()
    ax.legend()
    ax.set_xlim(0, 200)
    ax.set_ylim(-0.1, 0.1)

def update_idx(idx):
    t = np.linspace(0, 2.0, num=spec.n + 1, endpoint=True)
    line0.set_ydata(y_data[idx + 1: idx + spec.n + 2])
    line1.set_ydata(pred_fun(y_data[idx - 3: idx + 1], t, idx))
    line2.set_ydata(y_data[idx + 1: idx + spec.n + 2])
    line3.set_ydata(pred_fun_check(y_data[idx - 1: idx + 1], t, idx))
    # for ax in axs:
    #     ax.grid()
    #     ax.legend()
    fig.canvas.draw_idle()
    
idx_slider = ipywidgets.widgets.IntSlider(value=idx, min=4, max=6000, step=20)
ipywidgets.interact(update_idx, idx=idx_slider)